
# Tier Random Sequence Inclusion Analysis

This notebook avoids the large arrow files and instead loads the FASTA inputs produced under `proteindb/data/tier_train` and `proteindb/data/tier_sequences_random`.

The two analyses mirror the request: map each tier-sample sequence back to the most specific train tier it belongs to, then compare overlaps between the random tier samples themselves.




## Setup

We rely only on the local FASTA files (`tierX_sequences*.fasta`) and standard Python libraries so the notebook is portable without extra dependencies.



In [ ]:

from collections import Counter
from pathlib import Path

DATA_ROOT = Path('proteindb') / 'data'
TRAIN_DIR = DATA_ROOT / 'tier_train'
RANDOM_DIR = DATA_ROOT / 'tier_sequences_random'
TRAIN_SUFFIXES = ['_sequences_train.fasta', '_sequences.fasta']
RANDOM_SUFFIXES = ['_sequences_random.fasta', '.fasta']



In [ ]:

def read_fasta(path: Path) -> list[str]:
    sequences: list[str] = []
    current: list[str] = []
    with open(path, 'r') as fh:
        for line in fh:
            stripped = line.strip()
            if not stripped:
                continue
            if stripped.startswith('>'):
                if current:
                    sequences.append(''.join(current))
                    current = []
                continue
            current.append(stripped)
    if current:
        sequences.append(''.join(current))
    return sequences


In [ ]:

tier_train_paths: dict[int, Path] = {}
for tier in range(1, 7):
    for suffix in TRAIN_SUFFIXES:
        candidate = TRAIN_DIR / f'tier{tier}{suffix}'
        if candidate.exists():
            tier_train_paths[tier] = candidate
            break
    else:
        raise FileNotFoundError(f'Missing tier_train tier {tier}')

tier_random_paths: dict[int, Path] = {}
for tier in range(1, 7):
    for suffix in RANDOM_SUFFIXES:
        candidate = RANDOM_DIR / f'tier{tier}{suffix}'
        if candidate.exists():
            tier_random_paths[tier] = candidate
            break
    else:
        raise FileNotFoundError(f'Missing random tier {tier}')

train_sequences = {tier: set(read_fasta(path)) for tier, path in sorted(tier_train_paths.items())}
random_lists = {tier: read_fasta(path) for tier, path in sorted(tier_random_paths.items())}
random_sets = {tier: set(lst) for tier, lst in random_lists.items()}

print('Train tiers loaded:')
for tier, path in tier_train_paths.items():
    print(f'  tier{tier}: {path.name} ({len(train_sequences[tier])} sequences)')
print('Random tiers loaded:')
for tier, path in tier_random_paths.items():
    print(f'  tier{tier}: {path.name} ({len(random_lists[tier])} sequences)')



In [ ]:

train_tiers = sorted(train_sequences)

def most_specific_tier(seq: str) -> int | None:
    for tier in reversed(train_tiers):
        if seq in train_sequences[tier]:
            return tier
    return None

membership: dict[int, Counter[int | None]] = {}
for random_tier, seqs in random_lists.items():
    membership[random_tier] = Counter(most_specific_tier(seq) for seq in seqs)

print('Membership counts (random tier -> membership tier):')
for tier in sorted(membership):
    total = sum(membership[tier].values())
    print(f'  random tier {tier} ({total} samples):')
    for mem_tier, count in sorted(membership[tier].items(), key=lambda kv: (kv[0] is None, kv[0] or 0)):
        label = 'missing' if mem_tier is None else f'tier{mem_tier}'
        pct = count / total * 100
        print(f'    {label}: {count} ({pct:.2f}%)')

unique_memberships = sorted({mem_tier if mem_tier is not None else -1 for counter in membership.values() for mem_tier in counter})
print('Membership matrix counts (rows=random tier, columns=membership tier):')
header = ['random_tier'] + [('missing' if m == -1 else f'tier{m}') for m in unique_memberships]
print('	'.join(header))
for tier in sorted(membership):
            row = [f'tier{tier}'] + [str(membership[tier].get(mem, 0)) for mem in unique_memberships]
            print('	'.join(row))



In [ ]:

        overlap_counts: dict[int, dict[int, int]] = {tier: {} for tier in random_sets}
        for src in sorted(random_sets):
            for tgt in sorted(random_sets):
                overlap_counts[src][tgt] = len(random_sets[src] & random_sets[tgt])

        print('
Overlap counts matrix (random tiers):')
        header = [''] + [f'tier{tgt}' for tgt in sorted(random_sets)]
        print('	'.join(header))
        for src in sorted(overlap_counts):
            row = [f'tier{src}'] + [str(overlap_counts[src][tgt]) for tgt in sorted(overlap_counts[src])]
            print('	'.join(row))

        print('
Overlap percentages per random tier relative to its sample size:')
        for src in sorted(random_lists):
            total = len(random_lists[src])
            row = [f'tier{src}'] + [f"{overlap_counts[src][tgt]/total*100:.1f}%" for tgt in sorted(overlap_counts[src])]
            print('	'.join(row))

        sample_overlap = sorted(random_sets[1] & random_sets[2])[:5]
        print('
Sample sequences in both tier1 and tier2 random sets (first 5):')
        for seq in sample_overlap:
            print(f'{seq[:60]}... len={len(seq)}')




## Observations
- The membership section above shows how much of each random tier sits in progressively more specific train tiers; the matrix gives a quick sweep of overlaps.
- The overlap matrices reveal that the random tiers are largely disjoint (each tier retains ≈1000 unique samples) but you can still spot the handful of shared sequences for debugging or pruning experiments.
- Run this notebook end-to-end once the standard library is available to capture the real-time values; the code is fully data-driven and should finish quickly because the random FASTA files are small.

